# 00 — Load all monthly billings to Gold and datamarts
Discover every `billing-YYYY-MM.parquet` in the GCS-backed External Volume, then run the maintained end-to-end monthly backfill. Each month passes through Bronze, the FOCUS Data Contract, Silver, Gold, all certified datamarts, reconciliation, and finally archival.

In [0]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is None:
    raise FileNotFoundError("Open this notebook from the FinOps Cloud Data Platform Git Folder.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [0]:
ENVIRONMENT = "dev"
ARCHIVE_AFTER_SUCCESS = "true"

dbutils.widgets.dropdown("environment", ENVIRONMENT, ["dev", "prod"])
dbutils.widgets.dropdown("archive_after_success", ARCHIVE_AFTER_SUCCESS, ["true", "false"])
ENVIRONMENT = dbutils.widgets.get("environment")
ARCHIVE_AFTER_SUCCESS = dbutils.widgets.get("archive_after_success").lower() == "true"
print(f"Environment: {ENVIRONMENT}; archive after success: {ARCHIVE_AFTER_SUCCESS}")

In [0]:
import re

from finops_cloud.config import load_config
from finops_cloud.pipelines.billing_backfill import month_range

config = load_config(ENVIRONMENT)
monthly_path = f"{config.source_volume}/monthly"
pattern = re.compile(r"billing-([0-9]{4}-(?:0[1-9]|1[0-2]))[.]parquet$")
entries = dbutils.fs.ls(monthly_path)
parquet_names = sorted(entry.name for entry in entries if entry.name.endswith(".parquet"))
invalid_names = [name for name in parquet_names if pattern.fullmatch(name) is None]
if invalid_names:
    raise ValueError(f"Unexpected monthly Parquet names: {invalid_names}")
months = sorted(pattern.fullmatch(name).group(1) for name in parquet_names)
if not months:
    raise FileNotFoundError(f"No billing-YYYY-MM.parquet found in {monthly_path}")
expected = month_range(months[0], months[-1])
if months != expected:
    missing = sorted(set(expected) - set(months))
    raise ValueError(f"Monthly sequence is incomplete; missing: {missing}")

START_MONTH, END_MONTH = months[0], months[-1]
print(f"Detected {len(months)} months: {START_MONTH} through {END_MONTH}")
for name in parquet_names:
    print(f"- {monthly_path}/{name}")

In [0]:
from finops_cloud.pipelines.billing_backfill import run

results = run(
    ENVIRONMENT,
    START_MONTH,
    END_MONTH,
    archive=ARCHIVE_AFTER_SUCCESS,
)
display(results)

In [0]:
from finops_cloud.runtime import get_spark

spark_session = get_spark(config.profile)
expected_status = "CLOSED" if ARCHIVE_AFTER_SUCCESS else "CLOSED_DATA_LOADED"
status_table = config.table("month_status", "ops")
statuses = (
    spark_session.table(status_table)
    .where("billing_month >= '{0}' AND billing_month <= '{1}'".format(START_MONTH, END_MONTH))
    .select("billing_month", "status", "updated_at")
    .orderBy("billing_month")
)
failed = statuses.where(f"status <> '{expected_status}'").count()
if statuses.count() != len(months) or failed:
    raise RuntimeError("Final month-status verification failed")
print(f"SUCCESS: {len(months)} months loaded through Gold and datamarts.")
display(statuses)
display(spark_session.sql(f"SHOW TABLES IN {config.schema('gold')}"))
display(spark_session.sql(f"SHOW TABLES IN {config.schema('datamart')}"))